# DeepEval + LangGraph Agent Evaluation Notebook

This notebook demonstrates:

- LangGraph Agent Workflow
- Vector Database Retrieval
- GPT-4o-mini Integration
- DeepEval Agent Evaluations
- GEval (LLM-as-a-Judge)
- PyTest-style AI Testing
- Beautiful Evaluation Reporting
- Colab-Friendly Best Practices

## 1. Install Dependencies

In [ ]:
# Install required libraries

!pip -q install -U langgraph
!pip -q install -U langchain
!pip -q install -U langchain-openai
!pip -q install -U langchain-community
!pip -q install -U deepeval
!pip -q install -U faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.2.15 requires langgraph<1.2.0,>=1.1.5, but you have langgraph 1.2.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## 2. Fix Common Colab Timeout Issues

In [ ]:
import os

os.environ["DEEPEVAL_DISABLE_TELEMETRY"] = "YES"

print("DeepEval Optimized")


DeepEval Optimized


## 3. Load OpenAI API Key

Add `OPENAI_API_KEY` in Colab Secrets before running this cell.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("OPENAI API KEY LOADED")


OPENAI API KEY LOADED


## 4. Import Required Libraries

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, END

from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings
)

from langchain_community.vectorstores import FAISS

from deepeval import assert_test

from deepeval.metrics import (
    AnswerRelevancyMetric,
    GEval
)

from deepeval.test_case import (
    LLMTestCase,
    SingleTurnParams
)


## 5. Create Models

In [ ]:
# Main LLM

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# Embedding Model

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Fast Evaluation Model

EVAL_MODEL = "gpt-4o-mini"

print("MODELS READY")


MODELS READY


## 6. Create Knowledge Base

In [ ]:
documents = [

    '''
    LangGraph is a framework used for
    stateful multi-agent workflows.
    ''',

    '''
    DeepEval is used for evaluating
    AI agents and LLM applications.
    ''',

    '''
    OpenAI Agents SDK supports
    tool calling and handoffs.
    '''
]

print("KNOWLEDGE BASE CREATED")


KNOWLEDGE BASE CREATED


## 7. Create Vector Database

In [ ]:
vectorstore = FAISS.from_texts(

    texts=documents,

    embedding=embedding_model

)

retriever = vectorstore.as_retriever(

    search_kwargs={"k": 2}

)

print("VECTOR DATABASE READY")


VECTOR DATABASE READY


## 8. Define Agent State

In [ ]:
class AgentState(TypedDict):

    question: str
    context: str
    answer: str
    steps: list


## 9. Create Retrieval Node

In [ ]:
def retrieve_node(state):

    question = state["question"]

    docs = retriever.invoke(question)

    context = "\n".join(
        [doc.page_content for doc in docs]
    )

    steps = state.get("steps", [])

    steps.append(
        "Retrieved relevant documents"
    )

    return {

        "context": context,
        "steps": steps

    }


## 10. Create Generation Node

In [ ]:
def generate_node(state):

    question = state["question"]
    context = state["context"]

    prompt = f'''
    Answer ONLY from provided context.

    Context:
    {context}

    Question:
    {question}
    '''

    response = llm.invoke(prompt)

    steps = state.get("steps", [])

    steps.append(
        "Generated final response"
    )

    return {

        "answer": response.content,
        "steps": steps

    }


## 11. Build LangGraph Workflow

In [ ]:
graph = StateGraph(AgentState)

graph.add_node(
    "retrieve",
    retrieve_node
)

graph.add_node(
    "generate",
    generate_node
)

graph.set_entry_point("retrieve")

graph.add_edge(
    "retrieve",
    "generate"
)

graph.add_edge(
    "generate",
    END
)

app = graph.compile()

print("LANGGRAPH AGENT READY")


LANGGRAPH AGENT READY


## 12. Run Agent

In [ ]:
question = "What is LangGraph?"

result = app.invoke({

    "question": question,
    "steps": []

})


## 13. Beautiful Output Printing

In [ ]:
print("\n")
print("=" * 70)
print("AGENT EXECUTION")
print("=" * 70)

print("\nQUESTION")
print("-" * 70)
print(question)

print("\nRETRIEVED CONTEXT")
print("-" * 70)
print(result["context"])

print("\nFINAL ANSWER")
print("-" * 70)
print(result["answer"])

print("\nAGENT STEPS")
print("-" * 70)

for i, step in enumerate(result["steps"], start=1):

    print(f"{i}. {step}")




AGENT EXECUTION

QUESTION
----------------------------------------------------------------------
What is LangGraph?

RETRIEVED CONTEXT
----------------------------------------------------------------------

    LangGraph is a framework used for
    stateful multi-agent workflows.
    

    DeepEval is used for evaluating
    AI agents and LLM applications.
    

FINAL ANSWER
----------------------------------------------------------------------
LangGraph is a framework used for stateful multi-agent workflows.

AGENT STEPS
----------------------------------------------------------------------
1. Retrieved relevant documents
2. Generated final response


## 14. Create DeepEval Test Case

In [ ]:
test_case = LLMTestCase(

    input=question,

    actual_output=result["answer"],

    expected_output='''
    LangGraph is a framework used for
    stateful multi-agent workflows.
    '''

)


## 15. Answer Relevancy Metric

In [ ]:
relevancy_metric = AnswerRelevancyMetric(

    threshold=0.7,

    model=EVAL_MODEL

)


## 16. Custom Agent Evaluation using GEval

In [ ]:
agent_metric = GEval(

    name="Agent Workflow Quality",

    criteria='''
    Check whether the AI agent:

    1. Retrieved useful information
    2. Followed logical workflow
    3. Generated correct answer
    4. Used proper reasoning
    ''',

    evaluation_params=[

        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT

    ],

    model=EVAL_MODEL

)


## 17. Run Evaluations

In [ ]:
print("\n")
print("=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)

relevancy_metric.measure(test_case)

print("\nANSWER RELEVANCY")
print("-" * 70)

print("Score  :", round(relevancy_metric.score, 2))
print("Passed :", relevancy_metric.success)

if hasattr(relevancy_metric, "reason"):

    print("Reason :")
    print(relevancy_metric.reason)

agent_metric.measure(test_case)

print("\nAGENT QUALITY")
print("-" * 70)

print("Score  :", round(agent_metric.score, 2))
print("Passed :", agent_metric.success)

if hasattr(agent_metric, "reason"):

    print("Reason :")
    print(agent_metric.reason)


Output()



EVALUATION RESULTS


Output()


ANSWER RELEVANCY
----------------------------------------------------------------------
Score  : 1.0
Passed : True
Reason :
The score is 1.00 because the response directly addresses the question about LangGraph without any irrelevant statements.



AGENT QUALITY
----------------------------------------------------------------------
Score  : 0.99
Passed : True
Reason :
The Actual Output accurately reflects the information defined in the Input and matches the Expected Output perfectly. It provides a clear and concise definition of LangGraph, demonstrating a coherent logical workflow and proper justification in its reasoning.


## 19. Final Summary Dashboard

In [ ]:
print("\n")
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f'''
Question:
{question}

------------------------------------------------------------

Generated Answer:
{result["answer"]}

------------------------------------------------------------

Relevancy Score:
{round(relevancy_metric.score, 2)}

------------------------------------------------------------

Agent Quality Score:
{round(agent_metric.score, 2)}

------------------------------------------------------------

PyTest Result:
{"PASSED" if relevancy_metric.success else "FAILED"}
''')

print("=" * 70)

print("\nPROJECT COMPLETED SUCCESSFULLY")




FINAL SUMMARY

Question:
What is LangGraph?

------------------------------------------------------------

Generated Answer:
LangGraph is a framework used for stateful multi-agent workflows.

------------------------------------------------------------

Relevancy Score:
1.0

------------------------------------------------------------

Agent Quality Score:
0.99

------------------------------------------------------------

PyTest Result:
PASSED


PROJECT COMPLETED SUCCESSFULLY
